
# TPC digital current → measured charge-density and primary-density proxies (v21 DC)

This notebook is the DC-input successor to `TPC_IBF_PrimaryDensity_Extraction_v20_QA_CONDITIONS`.

It reads the new synchronized DC ROOT files directly, using the **positive pedestal-subtracted** quantity

\[
Q_{60}^{+}=\max(0,\mathrm{current}-60\,N_{\rm samples})
\]

as the default measured signal. The raw, signed, positive, and strict variants are all retained for QA.

The important change is that the geometry is taken from the actual `R1/R2/R3_ChannelMapping.csv` files. This gives exact pad-center radii and phi positions, identifies the four R1 antenna-pad rings, and lets us construct a pitch-cell area without pretending that the coarse `InfoTPC.h` histogram bins are the physical pad area.

The notebook deliberately separates:

- **measured DC density proxy**: pedestal-subtracted DC divided by `nsamples` and by the geometrical pitch-cell area;
- **standard-pad primary proxy**: measured density divided by the existing sector × TPC-layer gain map;
- **antenna-pad density**: kept separately because the normal 48-layer gain map does not calibrate the antenna pads;
- **measured**, **masked/clean**, and **noise QA** products.

The exact copper zig-zag polygon / inter-zig-zag metallization gap is *not* contained in the channel-mapping CSV. The first correction here therefore uses the **charge-collection pitch cell** (radial pitch × phi pitch), which is the correct zeroth-order collection area. A true copper-active-area correction can be added later from the Eagle `.brd` pad-plane geometry if it is shown to matter.


In [ ]:

import ROOT as root
import numpy as np
import pandas as pd
import math, os
from pathlib import Path
from array import array
from scipy import ndimage

root.gROOT.SetBatch(True)
root.TH1.AddDirectory(False)
root.gStyle.SetOptStat(0)

# ============================================================
# USER CONFIGURATION
# ============================================================

INPUT_ROOT = os.environ.get(
    "TPC_DC_INPUT",
    "dc_samples_run81558/dc_run81558_3samples.root"
)

# Directory containing R1_ChannelMapping.csv, R2_ChannelMapping.csv,
# R3_ChannelMapping.csv. Point this to 1_dc_preprocessing if the
# notebook lives elsewhere.
MAPPING_DIR = Path(os.environ.get("TPC_DC_MAPPING_DIR", "."))

# Main physics candidate. Keep the other variants as QA.
DC_VARIANT = os.environ.get("TPC_DC_VARIANT", "ped60_positive")
DC_VARIANTS = ("raw", "ped60_signed", "ped60_positive", "ped60_strict")

GAIN_MAP_FILE = Path(os.environ.get(
    "TPC_DC_GAIN_MAP",
    "../distort/input/layer_gain_79513_Mariia_side01.root"
))
GAIN_MAP_HIST_SIDE0 = "hGainMap_side0_South"
GAIN_MAP_HIST_SIDE1 = "hGainMap_side1_North"
GAIN_LAYER_OFFSET = 7
APPLY_STANDARD_PAD_GAIN = True

# Antenna pads are not represented by the normal 48-layer gain map.
# Keep them uncalibrated in primary-density products unless a dedicated
# calibration is supplied later.
APPLY_ANTENNA_GAIN = False

OUTPUT_DIR = Path(os.environ.get("TPC_DC_OUTPUT_DIR", "output_dc"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT = OUTPUT_DIR / "TPC_DC_primary_density_v21.root"

# Fine output grid used for QA / PHGarfield-oriented source maps.
GLOBAL_PHI_BINS = 720
PHI_MIN, PHI_MAX = -math.pi, math.pi
SOURCE_R_MIN_CM = 22.8
SOURCE_R_MAX_CM = 75.43
GLOBAL_DR_CM = 0.10

# Pad-plane active radial boundaries used by the mapping study.
# These preserve the radial gaps between R1/R2/R3 rather than
# using midpoint edges across module boundaries.
MODULE_ACTIVE_BOUNDS_CM = {
    0: (31.105, 40.249),
    1: (41.153, 57.475),
    2: (58.367, 75.911),
}

# R1 antenna + zig-zag are treated as one continuous R1 collection system.
R1_INNER_COLLECTION_CM = 22.8

# Conservative noise masks. They are saved separately and never destroy
# the measured maps.
MIN_EXPOSURE_FRACTION = 0.50
LOCAL_DEAD_FACTOR = 0.18
LOCAL_HOT_FACTOR = 4.0
LOCAL_PAD_RADIUS = 2
LOCAL_LAYER_RADIUS = 1
MIN_LOCAL_NEIGHBORS = 5
ANTENNA_LOG_MAD_NSIGMA = 6.0

print("Input ROOT:", INPUT_ROOT)
print("Mapping dir:", MAPPING_DIR)
print("Main DC variant:", DC_VARIANT)
print("Output:", OUTPUT_ROOT)



## 1. Load the pad mapping and derive the real pitch geometry

The mapping contains four special R1 antenna rows (`Radius = -4,-3,-2,-1`, names beginning with `DC.`), followed by the 16 normal R1 zig-zag rows (`Radius = 0...15`). R2 and R3 each contain 16 zig-zag rows.

For density, we use the **collection pitch cell**:

\[
A_{\rm cell} \simeq r\,\Delta\phi\,\Delta r.
\]

`Δφ` comes from the actual pad-center spacing in each row. `Δr` comes from midpoint boundaries inside an active module, with explicit module boundaries so the R1/R2 and R2/R3 gaps are preserved.


In [ ]:

def load_mapping(region):
    p = MAPPING_DIR / f"{region}_ChannelMapping.csv"
    if not p.exists():
        raise FileNotFoundError(
            f"Missing {p}. Set TPC_DC_MAPPING_DIR to the "
            "DC_ML_processing/1_dc_preprocessing directory."
        )
    df = pd.read_csv(p)
    required = {"Radius","Pad","PadName","PadR","PadPhi","FEE","FEE_Chan"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"{p} missing columns: {sorted(missing)}")
    return df

mapping = {
    0: load_mapping("R1"),
    1: load_mapping("R2"),
    2: load_mapping("R3"),
}

N_PADS = {0:94, 1:128, 2:192}
SECTOR_PITCH = 2*math.pi/12

def wrap_phi(phi):
    return (np.asarray(phi) + math.pi) % (2*math.pi) - math.pi

def row_geometry(df, radius_index):
    r = df[df["Radius"] == radius_index].copy().sort_values("Pad")
    ph = np.unwrap(r["PadPhi"].to_numpy(float))
    d = np.diff(ph)
    dphi = float(np.median(np.abs(d))) if len(d) else np.nan
    return {
        "radius_index": int(radius_index),
        "r_cm": float(np.mean(r["PadR"])) / 10.0,
        "n_pads": int(len(r)),
        "dphi": dphi,
        "phi_base": r["PadPhi"].to_numpy(float),
        "local_pad": r["Pad"].to_numpy(int),
        "pad_name": r["PadName"].astype(str).to_numpy(),
    }

zigzag_rows = {}
for module in range(3):
    zigzag_rows[module] = [row_geometry(mapping[module], il) for il in range(16)]

antenna_rows = [row_geometry(mapping[0], il) for il in (-4,-3,-2,-1)]

# Combined R1 radial cell edges: 4 antenna rings + 16 zig-zag rows.
r1_rows = antenna_rows + zigzag_rows[0]
r1_centers = np.array([x["r_cm"] for x in r1_rows])
r1_edges = np.empty(len(r1_centers)+1)
r1_edges[0] = R1_INNER_COLLECTION_CM
r1_edges[1:-1] = 0.5*(r1_centers[:-1] + r1_centers[1:])
r1_edges[-1] = MODULE_ACTIVE_BOUNDS_CM[0][1]

def module_edges(module):
    if module == 0:
        # Return only the 16 zig-zag portions of the combined R1 cells.
        return r1_edges[4:]
    centers = np.array([x["r_cm"] for x in zigzag_rows[module]])
    lo, hi = MODULE_ACTIVE_BOUNDS_CM[module]
    e = np.empty(len(centers)+1)
    e[0], e[-1] = lo, hi
    e[1:-1] = 0.5*(centers[:-1] + centers[1:])
    return e

zigzag_radial_edges = {m: module_edges(m) for m in range(3)}
antenna_radial_edges = r1_edges[:5]

def row_active_gap_deg(row):
    return (SECTOR_PITCH - row["n_pads"]*row["dphi"])*180/math.pi

print("Antenna rings:")
for i,row in enumerate(antenna_rows):
    dr = antenna_radial_edges[i+1]-antenna_radial_edges[i]
    print(f"  A{i}: r={row['r_cm']:.4f} cm, pads/sector={row['n_pads']}, "
          f"dphi={row['dphi']:.6f} rad, dr_cell={dr:.4f} cm, "
          f"sector gap={row_active_gap_deg(row):.3f} deg")

print("\nZig-zag regions:")
for m in range(3):
    centers=np.array([x["r_cm"] for x in zigzag_rows[m]])
    dphi=np.median([x["dphi"] for x in zigzag_rows[m]])
    dr=np.diff(zigzag_radial_edges[m])
    print(f"  R{m+1}: pads/sector={zigzag_rows[m][0]['n_pads']}, "
          f"r={centers[0]:.4f}..{centers[-1]:.4f} cm, "
          f"median dphi={dphi:.7f} rad, "
          f"median dr={np.median(dr):.4f} cm, "
          f"sector gap={row_active_gap_deg(zigzag_rows[m][0]):.3f} deg")

print("\nModule radial gaps:")
print("  R1->R2:", MODULE_ACTIVE_BOUNDS_CM[1][0]-MODULE_ACTIVE_BOUNDS_CM[0][1], "cm")
print("  R2->R3:", MODULE_ACTIVE_BOUNDS_CM[2][0]-MODULE_ACTIVE_BOUNDS_CM[1][1], "cm")



### Geometry information available from the mapping

The DC mapping files are already rich enough for the first-order area correction:

| component | rows | pads / sector / row | first→last row-center radius | typical $\Delta\phi$ |
|---|---:|---:|---:|---:|
| R1 antenna (`DC.*`) | 4 | 8 | 23.28 → 30.06 cm | 0.05975 rad |
| R1 zig-zag (`ZZ.*`) | 16 | 94 | 31.50 → 39.99 cm | 0.005307 rad |
| R2 zig-zag | 16 | 128 | 41.66 → 56.97 cm | 0.003959 rad |
| R3 zig-zag | 16 | 192 | 58.91 → 75.37 cm | 0.002651 rad |

The pad-plane active module limits used here are R1 = 31.105–40.249 cm,
R2 = 41.153–57.475 cm, and R3 = 58.367–75.911 cm. Therefore the
radial inactive gaps are about 0.904 cm between R1/R2 and 0.892 cm
between R2/R3.

Within a row, the mapping gives every pad center, so the local pad pitch
and the sector-edge uncovered interval can be calculated exactly from
the centers. What the CSV does **not** encode is the detailed copper
zig-zag polygon width/gap inside one pitch cell. That is a separate
second-order response correction; the Eagle pad-plane `.brd` files are
the appropriate source if we decide to implement it.



## 2. Read the DC ROOT format

For every pad/bin we keep both the integrated charge and the exposure.

The basic normalized signal is

\[
\bar q_{60}^{+}
=
\frac{\sum \max(0,\mathrm{current}-60N_{\rm samples})}
     {\sum N_{\rm samples}}.
\]

This prevents missing packets/channels from being interpreted as lower physical charge.


In [ ]:

fin = root.TFile.Open(str(INPUT_ROOT), "READ")
if not fin or fin.IsZombie():
    raise OSError(f"Cannot open {INPUT_ROOT}")

def th2_to_numpy(h):
    nx, ny = h.GetNbinsX(), h.GetNbinsY()
    z = np.empty((ny,nx),float)
    for iy in range(ny):
        for ix in range(nx):
            z[iy,ix] = h.GetBinContent(ix+1,iy+1)
    return z

def project_xy_numpy(h3):
    h2 = h3.Project3D("yx")
    h2.SetDirectory(0)
    # Project3D("yx") gives x = original Y (r), y = original X (phi)
    # Re-read explicitly into [r,phi] from the original TH3 instead to avoid
    # relying on ROOT's projection-axis naming.
    nr, nphi = h3.GetNbinsY(), h3.GetNbinsX()
    out=np.zeros((nr,nphi),float)
    for ir in range(1,nr+1):
        for ip in range(1,nphi+1):
            s=0.0
            for iz in range(1,h3.GetNbinsZ()+1):
                s += h3.GetBinContent(ip,ir,iz)
            out[ir-1,ip-1]=s
    redges=np.array([h3.GetYaxis().GetBinLowEdge(i) for i in range(1,nr+1)] +
                    [h3.GetYaxis().GetBinUpEdge(nr)],float)
    pedges=np.array([h3.GetXaxis().GetBinLowEdge(i) for i in range(1,nphi+1)] +
                    [h3.GetXaxis().GetBinUpEdge(nphi)],float)
    return out, redges, pedges

def safe_ratio(num,den):
    return np.divide(num,den,out=np.full_like(num,np.nan,dtype=float),
                     where=np.isfinite(den)&(den>0))

def global_name(variant,side):
    return f"h_dc_phi_r_frame_{variant}_side{side}"

def native_name(variant,side,sector,module):
    return f"h_dc_pad_vs_layer_{variant}_side{side}_sec{sector:02d}_R{module+1}"

# Global maps include antenna pads.
global_dc={}
for side in range(2):
    h_exp=fin.Get(f"h_dc_nsamples_phi_r_frame_side{side}")
    h_cnt=fin.Get(f"h_dc_sampleblocks_phi_r_frame_side{side}")
    if not h_exp or not h_cnt:
        raise KeyError(f"Missing global DC exposure/count histograms for side {side}")
    exp, r_edges_mm, phi_edges_02pi = project_xy_numpy(h_exp)
    cnt, _, _ = project_xy_numpy(h_cnt)

    item={"nsamples":exp,"sampleblocks":cnt,
          "r_edges_mm":r_edges_mm,"phi_edges_02pi":phi_edges_02pi}
    for variant in DC_VARIANTS:
        h=fin.Get(global_name(variant,side))
        if not h:
            raise KeyError(global_name(variant,side))
        q,_,_=project_xy_numpy(h)
        item[variant]=q
        item[f"{variant}_rate"]=safe_ratio(q,exp)
    global_dc[side]=item

# Native maps keep full pad x layer resolution for the 48 normal zig-zag layers.
native_dc={}
for side in range(2):
    for sector in range(12):
        for module in range(3):
            he=fin.Get(f"h_dc_nsamples_pad_vs_layer_side{side}_sec{sector:02d}_R{module+1}")
            hc=fin.Get(f"h_dc_sampleblocks_pad_vs_layer_side{side}_sec{sector:02d}_R{module+1}")
            if not he or not hc:
                raise KeyError(f"Missing native exposure/count side={side} sector={sector} R{module+1}")
            exp=th2_to_numpy(he)
            cnt=th2_to_numpy(hc)
            item={"nsamples":exp,"sampleblocks":cnt}
            for variant in DC_VARIANTS:
                h=fin.Get(native_name(variant,side,sector,module))
                if not h:
                    raise KeyError(native_name(variant,side,sector,module))
                q=th2_to_numpy(h)
                item[variant]=q
                item[f"{variant}_rate"]=safe_ratio(q,exp)
            native_dc[(side,module,sector)] = item

# Additive global exposure metadata.
def norm_value(side,label):
    h=fin.Get("h_dc_norm")
    if not h: return np.nan
    y=None
    for iy in range(1,h.GetNbinsY()+1):
        if h.GetYaxis().GetBinLabel(iy)==label:
            y=iy; break
    return h.GetBinContent(side+1,y) if y else np.nan

LIVE_FRAME_SLOTS=np.array([norm_value(s,"LIVE_FRAME_SLOTS") for s in range(2)])
LIVE_BCO_TICKS=np.array([norm_value(s,"LIVE_BCO_TICKS") for s in range(2)])

pGem=fin.Get("p_dc_gem_r1_current")
RUN_GEM_CURRENT=np.array([
    pGem.GetBinContent(1) if pGem else np.nan,
    pGem.GetBinContent(2) if pGem else np.nan
])

print("Live frame slots:",LIVE_FRAME_SLOTS)
print("R1 GEM current/load [South,North]:",RUN_GEM_CURRENT)



## 3. Preserve the noise information

We do **not** discard the signed or strict maps.

Three especially useful diagnostics are:

1. `signed < 0`: catches pathological pedestal-subtracted bins such as the single large negative side-0 bin seen in the test;
2. `strict / positive`: shows where the old upper cut at 30000 removes real signal; this is especially important for antenna pads;
3. low `nsamples` exposure: catches incompletely observed pads.

The clean mask below is conservative. It masks obvious readout/exposure/local outliers but leaves the measured map untouched.


In [ ]:

def local_reference(a, il, ip, layer_radius=LOCAL_LAYER_RADIUS, pad_radius=LOCAL_PAD_RADIUS):
    vals=[]
    for jl in range(max(0,il-layer_radius),min(a.shape[0],il+layer_radius+1)):
        for jp in range(max(0,ip-pad_radius),min(a.shape[1],ip+pad_radius+1)):
            if jl==il and jp==ip: continue
            v=a[jl,jp]
            if np.isfinite(v) and v>0:
                vals.append(v)
    if len(vals)<MIN_LOCAL_NEIGHBORS:
        return np.nan
    return float(np.median(vals))

native_masks={}
for key,item in native_dc.items():
    rate=item[f"{DC_VARIANT}_rate"]
    signed=item["ped60_signed_rate"]
    exp=item["nsamples"]

    finite_exp=exp[np.isfinite(exp)&(exp>0)]
    exp_ref=float(np.median(finite_exp)) if len(finite_exp) else np.nan
    lowexp=(~np.isfinite(exp)) | (exp<=0)
    if np.isfinite(exp_ref):
        lowexp |= exp < MIN_EXPOSURE_FRACTION*exp_ref

    negative=np.isfinite(signed)&(signed<0)
    spatial=np.zeros_like(rate,dtype=bool)

    for il in range(rate.shape[0]):
        for ip in range(rate.shape[1]):
            if lowexp[il,ip]:
                continue
            ref=local_reference(rate,il,ip)
            v=rate[il,ip]
            if np.isfinite(ref) and ref>0 and np.isfinite(v):
                if v < LOCAL_DEAD_FACTOR*ref or v > LOCAL_HOT_FACTOR*ref:
                    spatial[il,ip]=True

    native_masks[key]={
        "low_exposure":lowexp,
        "negative_signed":negative,
        "spatial_outlier":spatial,
        "combined":lowexp|negative|spatial,
    }

print("Native masked fractions:")
for side in range(2):
    vals=[]
    for module in range(3):
        for sector in range(12):
            m=native_masks[(side,module,sector)]["combined"]
            vals.append(np.mean(m))
    print(f"  side {side}: {np.mean(vals):.4%}")



## 4. Standard zig-zag gain correction

The existing gain map applies only to the 48 normal TPC layers. This is consistent with standard reconstruction, where antenna pads are excluded from the normal TPC layer range.

For each normal pad:

\[
\rho_{\rm DC}^{\rm meas}
=
\frac{\bar q_{60}^{+}}{A_{\rm pitch-cell}},
\qquad
\rho_{\rm primary}^{\rm proxy}
=
\frac{\rho_{\rm DC}^{\rm meas}}{G_{\rm sector,layer}}.
\]

The antenna pads are handled separately in the next section.


In [ ]:

gain_hist={}
if APPLY_STANDARD_PAD_GAIN:
    fg=root.TFile.Open(str(GAIN_MAP_FILE),"READ")
    if not fg or fg.IsZombie():
        raise OSError(f"Cannot open gain map {GAIN_MAP_FILE}")
    for side,name in [(0,GAIN_MAP_HIST_SIDE0),(1,GAIN_MAP_HIST_SIDE1)]:
        h=fg.Get(name)
        if not h: raise KeyError(name)
        h=h.Clone(name+"_clone"); h.SetDirectory(0)
        gain_hist[side]=h
    fg.Close()

def gain_for(side,sector,module,local_layer):
    if not APPLY_STANDARD_PAD_GAIN:
        return 1.0
    glayer=7+16*module+local_layer
    h=gain_hist[side]
    bx=h.GetXaxis().FindBin(sector+0.5)
    by=h.GetYaxis().FindBin((glayer-GAIN_LAYER_OFFSET)+0.5)
    g=float(h.GetBinContent(bx,by))
    return g if np.isfinite(g) and g>0 else np.nan

phi_out_edges=np.linspace(PHI_MIN,PHI_MAX,GLOBAL_PHI_BINS+1)
phi_out_centers=0.5*(phi_out_edges[:-1]+phi_out_edges[1:])
r_out_edges=np.arange(SOURCE_R_MIN_CM,SOURCE_R_MAX_CM+GLOBAL_DR_CM*1.0001,GLOBAL_DR_CM)
if r_out_edges[-1] < SOURCE_R_MAX_CM:
    r_out_edges=np.r_[r_out_edges,SOURCE_R_MAX_CM]
r_out_edges[-1]=SOURCE_R_MAX_CM
r_out_centers=0.5*(r_out_edges[:-1]+r_out_edges[1:])

shape=(len(r_out_centers),len(phi_out_centers))

def circular_delta(a,b):
    return np.arctan2(np.sin(a-b),np.cos(a-b))

def fill_cell(sum_map,count_map,value,rlo,rhi,phi_center,dphi):
    if not np.isfinite(value): return
    ir=np.where((r_out_centers>=rlo)&(r_out_centers<rhi))[0]
    ip=np.where(np.abs(circular_delta(phi_out_centers,phi_center)) <= 0.5*dphi)[0]
    if not len(ir) or not len(ip): return
    sum_map[np.ix_(ir,ip)] += value
    count_map[np.ix_(ir,ip)] += 1.0

def fill_cell_sum_only(sum_map,value,rlo,rhi,phi_center,dphi):
    """Accumulate another quantity into exactly the same geometrical cell
    without incrementing the shared coverage counter a second time."""
    if not np.isfinite(value): return
    ir=np.where((r_out_centers>=rlo)&(r_out_centers<rhi))[0]
    ip=np.where(np.abs(circular_delta(phi_out_centers,phi_center)) <= 0.5*dphi)[0]
    if not len(ir) or not len(ip): return
    sum_map[np.ix_(ir,ip)] += value

dc_measured_std={s:np.zeros(shape) for s in range(2)}
primary_measured_std={s:np.zeros(shape) for s in range(2)}
dc_clean_std={s:np.zeros(shape) for s in range(2)}
primary_clean_std={s:np.zeros(shape) for s in range(2)}
std_count={s:np.zeros(shape) for s in range(2)}
std_clean_count={s:np.zeros(shape) for s in range(2)}

for side in range(2):
    for module in range(3):
        redges=zigzag_radial_edges[module]
        for sector in range(12):
            item=native_dc[(side,module,sector)]
            mask=native_masks[(side,module,sector)]["combined"]
            rate=item[f"{DC_VARIANT}_rate"]

            for il,row in enumerate(zigzag_rows[module]):
                r=row["r_cm"]
                rlo,rhi=redges[il],redges[il+1]
                dr=rhi-rlo
                dphi=row["dphi"]
                area=r*dphi*dr
                g=gain_for(side,sector,module,il)

                for pad in range(row["n_pads"]):
                    v=rate[il,pad]
                    if not np.isfinite(v): continue
                    phi=float(wrap_phi(row["phi_base"][pad] + sector*SECTOR_PITCH))
                    rho=v/area
                    prho=rho/g if np.isfinite(g) and g>0 else np.nan

                    fill_cell(dc_measured_std[side],std_count[side],rho,rlo,rhi,phi,dphi)
                    fill_cell_sum_only(primary_measured_std[side],prho,rlo,rhi,phi,dphi)

                    if not mask[il,pad]:
                        fill_cell(dc_clean_std[side],std_clean_count[side],rho,rlo,rhi,phi,dphi)
                        fill_cell_sum_only(primary_clean_std[side],prho,rlo,rhi,phi,dphi)

for side in range(2):
    for d,c in [(dc_measured_std[side],std_count[side]),
                (primary_measured_std[side],std_count[side]),
                (dc_clean_std[side],std_clean_count[side]),
                (primary_clean_std[side],std_clean_count[side])]:
        good=c>0
        d[good]/=c[good]
        d[~good]=np.nan



## 5. R1 antenna pads

The mapping gives four antenna rings with 8 pads per sector, at approximately 23.28, 25.54, 27.80, and 30.06 cm. These pads are present in the DC global histogram but are not part of the normal 48-layer native maps.

For each antenna pad we sample the global DC histogram at its mapped `(r,phi)` position and divide by its `nsamples` exposure and its pitch-cell area.

No normal TPC gain correction is applied. The antenna response is therefore kept as a **separate primary proxy / calibration diagnostic**, rather than silently forcing it onto the 48-layer gain scale.


In [ ]:

def global_bin_value(item, arrname, r_cm, phi):
    # Input global histogram uses phi in [0,2pi), r in mm.
    ph=phi%(2*math.pi)
    redges=item["r_edges_mm"]
    pedges=item["phi_edges_02pi"]
    ir=np.searchsorted(redges,r_cm*10.0,side="right")-1
    ip=np.searchsorted(pedges,ph,side="right")-1
    if ir<0 or ir>=len(redges)-1 or ip<0 or ip>=len(pedges)-1:
        return np.nan
    return item[arrname][ir,ip]

antenna_records={0:[],1:[]}

for side in range(2):
    gitem=global_dc[side]
    for ia,row in enumerate(antenna_rows):
        r=row["r_cm"]
        rlo,rhi=antenna_radial_edges[ia],antenna_radial_edges[ia+1]
        dr=rhi-rlo
        dphi=row["dphi"]
        area=r*dphi*dr

        for sector in range(12):
            for pad in range(row["n_pads"]):
                phi=float(wrap_phi(row["phi_base"][pad]+sector*SECTOR_PITCH))
                qpos=global_bin_value(gitem,"ped60_positive",r,phi)
                qsigned=global_bin_value(gitem,"ped60_signed",r,phi)
                qstrict=global_bin_value(gitem,"ped60_strict",r,phi)
                exp=global_bin_value(gitem,"nsamples",r,phi)
                cnt=global_bin_value(gitem,"sampleblocks",r,phi)

                rate=qpos/exp if np.isfinite(exp) and exp>0 else np.nan
                signed_rate=qsigned/exp if np.isfinite(exp) and exp>0 else np.nan
                strict_rate=qstrict/exp if np.isfinite(exp) and exp>0 else np.nan

                antenna_records[side].append({
                    "ring":ia,"sector":sector,"pad":pad,
                    "r_cm":r,"rlo":rlo,"rhi":rhi,
                    "phi":phi,"dphi":dphi,"area_cm2":area,
                    "rate":rate,
                    "signed_rate":signed_rate,
                    "strict_rate":strict_rate,
                    "nsamples":exp,
                    "sampleblocks":cnt,
                    "density":rate/area if np.isfinite(rate) else np.nan,
                })

def mark_antenna_noise(records):
    df=pd.DataFrame(records)
    df["noise_mask"]=False

    # Obvious signed-pedestal pathology / missing exposure.
    df.loc[(~np.isfinite(df["nsamples"])) | (df["nsamples"]<=0), "noise_mask"]=True
    df.loc[np.isfinite(df["signed_rate"]) & (df["signed_rate"]<0), "noise_mask"]=True

    # Robust ring-by-ring log-density outlier flag. This is deliberately loose.
    for ring in sorted(df["ring"].unique()):
        sel=(df["ring"]==ring)
        v=df.loc[sel,"density"].to_numpy(float)
        good=np.isfinite(v)&(v>0)
        if np.count_nonzero(good)<8: continue
        lv=np.log(v[good])
        med=np.median(lv)
        mad=1.4826*np.median(np.abs(lv-med))
        if mad<=0: continue
        bad=np.zeros_like(v,dtype=bool)
        bad[good]=np.abs(np.log(v[good])-med)>ANTENNA_LOG_MAD_NSIGMA*mad
        idx=df.index[sel]
        df.loc[idx[bad],"noise_mask"]=True
    return df

antenna_df={s:mark_antenna_noise(antenna_records[s]) for s in range(2)}

for side in range(2):
    print(f"side {side} antenna:")
    for ring in range(4):
        d=antenna_df[side]
        s=d[d.ring==ring]
        med=np.nanmedian(s["density"])
        nbad=int(np.count_nonzero(s["noise_mask"]))
        print(f"  ring {ring}: median density={med:.6g}, masked={nbad}/{len(s)}")



## 6. Combine standard pads and antenna pads, while preserving measured and clean products

The combined measured map keeps all positive DC measurements.

The combined clean map masks only the conservative noise flags. It does **not** smooth across sector boundaries or module gaps. Missing/gap regions remain `NaN`.

A separate `antenna_mask` is written so the uncalibrated antenna response is always identifiable.


In [ ]:

dc_measured={s:dc_measured_std[s].copy() for s in range(2)}
dc_clean={s:dc_clean_std[s].copy() for s in range(2)}
primary_measured={s:primary_measured_std[s].copy() for s in range(2)}
primary_clean={s:primary_clean_std[s].copy() for s in range(2)}

antenna_mask_map={s:np.zeros(shape,dtype=bool) for s in range(2)}
noise_mask_map={s:np.zeros(shape,dtype=bool) for s in range(2)}

for side in range(2):
    # We fill antennas into otherwise empty low-r cells.
    sum_meas=np.zeros(shape); cnt_meas=np.zeros(shape)
    sum_clean=np.zeros(shape); cnt_clean=np.zeros(shape)
    amask=np.zeros(shape,float)

    for _,rec in antenna_df[side].iterrows():
        rho=rec["density"]
        if not np.isfinite(rho): continue
        rlo,rhi=float(rec["rlo"]),float(rec["rhi"])
        phi,dphi=float(rec["phi"]),float(rec["dphi"])

        fill_cell(sum_meas,cnt_meas,rho,rlo,rhi,phi,dphi)
        fill_cell(amask,np.zeros_like(amask),1.0,rlo,rhi,phi,dphi)

        if not bool(rec["noise_mask"]):
            fill_cell(sum_clean,cnt_clean,rho,rlo,rhi,phi,dphi)
        else:
            tmp=np.zeros(shape); tc=np.zeros(shape)
            fill_cell(tmp,tc,1.0,rlo,rhi,phi,dphi)
            noise_mask_map[side] |= tc>0

    good=cnt_meas>0
    antenna_density=np.full(shape,np.nan)
    antenna_density[good]=sum_meas[good]/cnt_meas[good]

    goodc=cnt_clean>0
    antenna_clean=np.full(shape,np.nan)
    antenna_clean[goodc]=sum_clean[goodc]/cnt_clean[goodc]

    antenna_mask_map[side] |= good

    # Fill only low-r cells that are not already standard zig-zag cells.
    take=good & ~np.isfinite(dc_measured[side])
    dc_measured[side][take]=antenna_density[take]
    primary_measured[side][take]=antenna_density[take]  # uncalibrated antenna primary proxy

    takec=goodc & ~np.isfinite(dc_clean[side])
    dc_clean[side][takec]=antenna_clean[takec]
    primary_clean[side][takec]=antenna_clean[takec]

# Explicitly preserve physical module gaps as NaN.
for side in range(2):
    for lo,hi in [(MODULE_ACTIVE_BOUNDS_CM[0][1],MODULE_ACTIVE_BOUNDS_CM[1][0]),
                  (MODULE_ACTIVE_BOUNDS_CM[1][1],MODULE_ACTIVE_BOUNDS_CM[2][0])]:
        m=(r_out_centers>=lo)&(r_out_centers<hi)
        dc_measured[side][m,:]=np.nan
        dc_clean[side][m,:]=np.nan
        primary_measured[side][m,:]=np.nan
        primary_clean[side][m,:]=np.nan

print("Combined maps built.")



## 7. Geometry/noise QA

The radial profile is intentionally shown with antenna points visible rather than forcing them onto the R1 zig-zag trend. A discontinuity here is useful information: it can indicate an antenna-area/gain response issue, a real low-radius physics effect, or residual noise.


In [ ]:

def robust_profile_r(a):
    out=np.full(a.shape[0],np.nan)
    for i,row in enumerate(a):
        v=row[np.isfinite(row)&(row>0)]
        if len(v): out[i]=np.median(v)
    return out

for side in range(2):
    print(f"\nSIDE {side}")
    for name,a in [
        ("DC measured",dc_measured[side]),
        ("DC clean",dc_clean[side]),
        ("Primary measured",primary_measured[side]),
        ("Primary clean",primary_clean[side]),
    ]:
        p=robust_profile_r(a)
        print(name, "finite radial bins =", np.count_nonzero(np.isfinite(p)))

    # Strict-loss diagnostic on the coarse global DC grid.
    pos=global_dc[side]["ped60_positive_rate"]
    strict=global_dc[side]["ped60_strict_rate"]
    loss=np.divide(pos-strict,pos,out=np.full_like(pos,np.nan),
                   where=np.isfinite(pos)&(pos>0))
    print("strict-loss median:",np.nanmedian(loss),
          "95%:",np.nanpercentile(loss[np.isfinite(loss)],95))

    signed=global_dc[side]["ped60_signed_rate"]
    print("global signed-negative bins:",np.count_nonzero(np.isfinite(signed)&(signed<0)))



## 8. Write ROOT products

`h_dc_density_*` is the measured avalanche/DC density proxy before gain division.

`h_primary_density_*` divides normal TPC pads by the existing gain map. Antenna cells remain explicitly marked by `h_antenna_mask_*` and are therefore **not claimed to be absolutely gain-calibrated primary density yet**.


In [ ]:

def array_to_th2(name,title,a):
    h=root.TH2D(name,title,
                len(phi_out_edges)-1,array("d",phi_out_edges),
                len(r_out_edges)-1,array("d",r_out_edges))
    h.SetDirectory(0)
    for ir in range(a.shape[0]):
        for ip in range(a.shape[1]):
            v=a[ir,ip]
            h.SetBinContent(ip+1,ir+1,float(v) if np.isfinite(v) else 0.0)
    return h

fout=root.TFile(str(OUTPUT_ROOT),"RECREATE")

for side in range(2):
    objs=[
        array_to_th2(f"h_dc_density_measured_rphi_side{side}",
                     f"Measured DC density proxy side {side};#phi;r [cm]",
                     dc_measured[side]),
        array_to_th2(f"h_dc_density_clean_rphi_side{side}",
                     f"Noise-masked DC density proxy side {side};#phi;r [cm]",
                     dc_clean[side]),
        array_to_th2(f"h_primary_density_measured_rphi_side{side}",
                     f"Primary-density proxy side {side};#phi;r [cm]",
                     primary_measured[side]),
        array_to_th2(f"h_primary_density_clean_rphi_side{side}",
                     f"Noise-masked primary-density proxy side {side};#phi;r [cm]",
                     primary_clean[side]),
        array_to_th2(f"h_antenna_mask_rphi_side{side}",
                     f"Antenna-pad coverage mask side {side};#phi;r [cm]",
                     antenna_mask_map[side].astype(float)),
        array_to_th2(f"h_dc_noise_mask_rphi_side{side}",
                     f"Conservative DC noise mask side {side};#phi;r [cm]",
                     noise_mask_map[side].astype(float)),
    ]
    for h in objs:
        h.Write()

# Metadata
hgem=root.TH1D("h_gem_current","R1 GEM current/load;side;load current",2,-0.5,1.5)
hgem.GetXaxis().SetBinLabel(1,"South")
hgem.GetXaxis().SetBinLabel(2,"North")
for s in range(2):
    hgem.SetBinContent(s+1,float(RUN_GEM_CURRENT[s]))
hgem.Write()

hexp=root.TH1D("h_live_frame_slots","Synchronized DC live exposure;side;frame slots",2,-0.5,1.5)
for s in range(2):
    hexp.SetBinContent(s+1,float(LIVE_FRAME_SLOTS[s]))
hexp.Write()

root.TNamed("dc_main_variant",DC_VARIANT).Write()
root.TNamed("dc_primary_definition",
            "Standard pads: (ped60_positive/nsamples)/(pitch-cell area)/gain. "
            "Antenna pads: (ped60_positive/nsamples)/(pitch-cell area), no normal 48-layer gain correction.").Write()
root.TNamed("dc_geometry_definition",
            "Pad-center pitch geometry from R1/R2/R3_ChannelMapping.csv; module radial gaps preserved.").Write()

fout.Close()
print("Wrote",OUTPUT_ROOT)



## Next calibration step

Before using the antenna points as an absolute primary-density constraint, compare their radial medians to the gain-corrected R1 zig-zag trend across several runs and GEM-current settings.

If the antenna/R1 ratio is stable with run and GEM current, it is likely an instrumental geometry/response factor and can be calibrated. If it changes systematically with space charge / GEM current, that is physics information and should **not** be divided away.

For an even more exact second-order geometry correction, the Eagle pad-plane `.brd` files in the sPHENIX calibration repository can be parsed to obtain the actual copper zig-zag polygons. That correction should be kept separate from the pitch-cell collection area used above.
